# 4 · Preparación del dataset para el modelado

**Proyecto:** Predicción de readmisión hospitalaria en pacientes diabéticos
**Dataset:** *Diabetes 130-US hospitals for years 1999-2008* (UCI Machine Learning Repository)
**Autor:** Diego Rodríguez Díaz del Campo

---

### Objetivo de este notebook

Convertir el dataset limpio en un conjunto de datos **listo para entrenar un modelo**,
aplicando las decisiones que el EDA dejó preparadas (apartados 5.6, 6.3 y 7 del
notebook 03). Cada transformación se justifica con la evidencia del EDA; nada se hace
por rutina.

El modelado **no** forma parte de este proyecto: este notebook termina con los
conjuntos `train` y `test` guardados en disco, que serán la entrada de un proyecto
futuro.

| | |
|---|---|
| **Entrada** | `data/processed/diabetic_data_clean.csv` (101.766 × 35) y `data/raw/diabetic_data.csv` (solo para recuperar `patient_nbr`) |
| **Variable objetivo** | `readmitted` |
| **Salida** | `data/processed/train.csv` y `data/processed/test.csv` |

### Contenido

1. Configuración, carga y recuperación de `patient_nbr`
2. Exclusión de pacientes fallecidos y en cuidados paliativos
3. Agrupación de categorías poco frecuentes y de alta cardinalidad
4. Eliminación de variables sin señal
5. Formulación de la variable objetivo
6. Codificación de las variables categóricas
7. División en `train` y `test` sin fuga de datos entre pacientes
8. Guardado y conclusiones

## 1 · Configuración, carga y recuperación de `patient_nbr`

### 1.1 Carga y recuperación de tipos

Se parte del dataset limpio (notebook 02). Como se explicó en el notebook 03, el CSV
**no conserva los `dtype`**: los tres códigos administrativos (`admission_type_id`,
`discharge_disposition_id`, `admission_source_id`) y los códigos de diagnóstico vuelven
a leerse como números y hay que declararlos de nuevo como categorías. Se repite aquí la
misma corrección que en el notebook 03, sin volver a discutirla.

Solo se importa lo que este apartado necesita (`pandas`, `numpy`, `pathlib`). Las
librerías gráficas y `scikit-learn` se importarán en el apartado en que hagan falta,
para que quede claro *para qué* se usa cada una.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Raíz del proyecto: los notebooks se ejecutan desde notebooks/, así que es la carpeta de encima
BASE = Path.cwd().parent
RAW = BASE / 'data' / 'raw'
PROC = BASE / 'data' / 'processed'

df = pd.read_csv(PROC / 'diabetic_data_clean.csv', low_memory=False)            

print(f'Dimensiones: {df.shape}')
df.head()

Dimensiones: (101766, 35)


,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,...,glipizide,glyburide,pioglitazone,rosiglitazone,acarbose,insulin,glyburide-metformin,change,diabetesMed,readmitted
0,Caucasian,Female,[0-10),6,25,1,1,Unknown,Pediatrics-Endocrinology,41,...,No,No,No,No,No,No,No,No,No,NO
1,Caucasian,Female,[10-20),1,1,7,3,Unknown,Unknown,59,...,No,No,No,No,No,Up,No,Ch,Yes,>30
2,AfricanAmerican,Female,[20-30),1,1,7,2,Unknown,Unknown,11,...,Steady,No,No,No,No,No,No,No,Yes,NO
3,Caucasian,Male,[30-40),1,1,7,2,Unknown,Unknown,44,...,No,No,No,No,No,Up,No,Ch,Yes,NO
4,Caucasian,Male,[40-50),1,1,7,1,Unknown,Unknown,51,...,Steady,No,No,No,No,Steady,No,Ch,Yes,NO


In [2]:
# Códigos administrativos: son etiquetas, no cantidades
columnas_categoricas_numericas = [
    'admission_type_id',
    'discharge_disposition_id',
    'admission_source_id'
]

df[columnas_categoricas_numericas] = df[columnas_categoricas_numericas].astype('category')

# Códigos de diagnóstico: el código 700 no es "mayor" que el 200
df[['diag_1', 'diag_2', 'diag_3']] = df[['diag_1', 'diag_2', 'diag_3']].astype('category')

df.dtypes

race                          object
gender                        object
age                           object
admission_type_id           category
discharge_disposition_id    category
admission_source_id         category
time_in_hospital               int64
payer_code                    object
medical_specialty             object
num_lab_procedures             int64
num_procedures                 int64
num_medications                int64
number_outpatient              int64
number_emergency               int64
number_inpatient               int64
diag_1                      category
diag_2                      category
diag_3                      category
number_diagnoses               int64
max_glu_serum                 object
A1Cresult                     object
metformin                     object
repaglinide                   object
nateglinide                   object
glimepiride                   object
glipizide                     object
glyburide                     object
p

### 1.2 Por qué hay que recuperar `patient_nbr`

En la limpieza se eliminó `patient_nbr` porque un identificador de paciente no es una
variable predictora. Sin embargo, el notebook 01 mostró que los 101.766 ingresos
corresponden a **71.518 pacientes distintos**: hay pacientes con 2, 3 y hasta 40
ingresos en el dataset.

Esto tiene una consecuencia directa sobre el apartado 7. Si se dividen las *filas* al
azar entre `train` y `test`, los ingresos de un mismo paciente quedarán repartidos en
ambos conjuntos. El modelo vería en `test` a pacientes que ya conoce de `train`, con sus
mismas características demográficas y clínicas, y su rendimiento parecería mejor de lo
que es en realidad. Es una forma de **fuga de datos** (*data leakage*): la división debe
hacerse **por paciente**, y para eso hace falta el identificador.

### Cómo se recupera: por posición, no por clave

El dataset limpio ya no tiene ninguna columna que sirva de clave para cruzar con el
original (`encounter_id` también se eliminó). Pero la limpieza **eliminó 15 columnas y
0 filas, y nunca reordenó**: la fila *i* del dataset limpio es la fila *i* del original.
Por tanto, `patient_nbr` se puede copiar del original **por posición**.

Es una hipótesis razonable, pero es una hipótesis. Antes de pegar la columna se comprueba
con dos pruebas:

1. Que ambos ficheros tienen exactamente **101.766 filas**.
2. Que una columna que sobrevivió intacta a la limpieza, `time_in_hospital`, **coincide
   fila a fila** en los dos ficheros.

Del original solo se leen las dos columnas necesarias (`usecols`): `patient_nbr`, que es
lo que se quiere recuperar, y `time_in_hospital`, que sirve de columna de control.

In [3]:
original = pd.read_csv(
    RAW / 'diabetic_data.csv',
    usecols=['patient_nbr', 'time_in_hospital']
)

print(f'Filas en el limpio:   {len(df)}')
print(f'Filas en el original: {len(original)}')

Filas en el limpio:   101766
Filas en el original: 101766


Las dos tablas tienen el mismo número de filas. Ahora la comprobación decisiva: que
`time_in_hospital` coincide **posición a posición**.

Se compara con `.to_numpy()` en vez de comparar las dos Series directamente. La razón es
que pandas, al operar entre dos Series, las **alinea por índice**; aquí interesa
justamente lo contrario, comparar por posición pura, porque es lo que se va a asumir al
pegar la columna. Con los arrays de NumPy la comparación es estrictamente posicional.

In [4]:
coinciden = (df['time_in_hospital'].to_numpy() == original['time_in_hospital'].to_numpy())

print(f'Filas que coinciden: {coinciden.sum()} de {len(coinciden)}')
print(f'¿Coinciden todas?    {coinciden.all()}')

Filas que coinciden: 101766 de 101766
¿Coinciden todas?    True


Las dos comprobaciones son favorables, así que se pega `patient_nbr` por posición. Se
inserta como **primera columna** con `insert(0, ...)`, porque es un identificador y no una
variable más: así queda claro a simple vista que no forma parte de las predictoras.

Como verificación final, el número de pacientes distintos debe ser **71.518**, la cifra
obtenida en el notebook 01 sobre el dataset original. Si diera otra cosa, la columna se
habría pegado desalineada.

In [5]:
df.insert(0, 'patient_nbr', original['patient_nbr'].to_numpy())

print(f'Dimensiones:        {df.shape}')
print(f'Pacientes distintos: {df["patient_nbr"].nunique()}')
df.head()

Dimensiones:        (101766, 36)
Pacientes distintos: 71518


,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,...,glipizide,glyburide,pioglitazone,rosiglitazone,acarbose,insulin,glyburide-metformin,change,diabetesMed,readmitted
0,8222157,Caucasian,Female,[0-10),6,25,1,1,Unknown,Pediatrics-Endocrinology,...,No,No,No,No,No,No,No,No,No,NO
1,55629189,Caucasian,Female,[10-20),1,1,7,3,Unknown,Unknown,...,No,No,No,No,No,Up,No,Ch,Yes,>30
2,86047875,AfricanAmerican,Female,[20-30),1,1,7,2,Unknown,Unknown,...,Steady,No,No,No,No,No,No,No,Yes,NO
3,82442376,Caucasian,Male,[30-40),1,1,7,2,Unknown,Unknown,...,No,No,No,No,No,Up,No,Ch,Yes,NO
4,42519267,Caucasian,Male,[40-50),1,1,7,1,Unknown,Unknown,...,Steady,No,No,No,No,Steady,No,Ch,Yes,NO


### Interpretación del apartado 1

Las tres verificaciones han sido favorables:

| Comprobación | Resultado esperado | Obtenido |
|---|---|---|
| Filas en ambos ficheros | 101.766 | 101.766 y 101.766 |
| `time_in_hospital` coincide fila a fila | todas | 101.766 de 101.766 (`True`) |
| Pacientes distintos tras pegar `patient_nbr` | 71.518 (notebook 01) | 71.518 |

La hipótesis de que la limpieza no alteró el orden de las filas queda **confirmada con
evidencia**, no asumida. El dataset de trabajo tiene ahora **101.766 × 36**: las 35
columnas del limpio más `patient_nbr` en primera posición.

Dos advertencias para el resto del notebook:

- `patient_nbr` es un **identificador**, no una variable predictora. Se conserva
  únicamente para hacer la división por paciente del apartado 7 y **no** debe entrar
  en el modelo: un número de historia clínica no dice nada del riesgo de readmisión.
- La cifra de 71.518 pacientes es válida **ahora**; cuando en el apartado 2 se excluyan
  los fallecidos y los pacientes en cuidados paliativos, cambiará y habrá que
  recalcularla.

## 2 · Exclusión de pacientes fallecidos y en cuidados paliativos

### El problema: un `NO` que no significa "no readmitido"

La variable objetivo `readmitted` toma el valor `NO` cuando el paciente no volvió a
ingresar. Pero hay un grupo de pacientes para los que ese `NO` **no es una decisión
clínica ni un buen resultado, sino una imposibilidad física**: los que fallecieron
durante el ingreso. Un paciente que muere en el hospital no puede ser readmitido.

Si estas filas se quedan en el dataset, el modelo aprendería que *"morir en el hospital
protege de la readmisión"*: un patrón numéricamente cierto y clínicamente absurdo. Y el
EDA ya mostró el efecto: el tramo `[90-100)` de `age` tenía un `NO` anormalmente alto
que se explicaba en parte por los fallecidos (apartado 5.5 del notebook 03).

### Qué códigos se excluyen y por qué

Según `IDS_mapping.csv`, `discharge_disposition_id` tiene cuatro códigos de fallecimiento
y dos de cuidados paliativos (*hospice*). Los recuentos son los del dataset limpio:

| Código | Descripción | Ingresos | `<30` / `>30` / `NO` |
|---|---|---|---|
| 11 | Expired | 1.642 | 0 / 0 / 1.642 |
| 19 | Expired at home. Medicaid only, hospice | 8 | 0 / 0 / 8 |
| 20 | Expired in a medical facility. Medicaid only, hospice | 2 | 0 / 0 / 2 |
| 21 | Expired, place unknown. Medicaid only, hospice | 0 | — |
| 13 | Hospice / home | 399 | 19 / 36 / 344 |
| 14 | Hospice / medical facility | 372 | 24 / 7 / 341 |
| | **Total** | **2.423** | |

Los tres códigos de fallecimiento presentes (11, 19, 20) son `NO` en el **100 %** de los
casos: la evidencia es inequívoca. El código 21 no aparece en el dataset, pero se incluye
en la lista para que la regla sea completa y no dependa de qué códigos hay hoy.

Los códigos de *hospice* (13, 14) son un caso distinto y merecen justificación propia.
El alta a cuidados paliativos implica un pronóstico de vida limitado y, sobre todo, un
**cambio de objetivo asistencial**: se renuncia a la intervención curativa, de modo que un
empeoramiento no conduce a un reingreso hospitalario sino a manejo del confort. No son
todos `NO` (86 % y 92 %), pero su `NO` mayoritario obedece a la misma lógica que el de los
fallecidos: **no informa de la calidad del alta ni del riesgo del paciente**, que es
exactamente lo que el modelo debe aprender. Por eso se excluyen junto con ellos.

### Se excluyen ingresos, no pacientes

Un paciente que falleció en su último ingreso puede tener ingresos anteriores en el
dataset. Esos ingresos anteriores son **válidos**: en ellos el paciente fue dado de alta
vivo y su `readmitted` refleja lo que realmente ocurrió después. Por tanto se eliminan
solo las **filas** con esos códigos de alta, no todas las filas de esos `patient_nbr`.

### Qué se comprueba

1. Que la máscara selecciona exactamente **2.423** filas (la cifra del contraste
   `[90-100)` del notebook 03).
2. Que tras filtrar quedan **99.343** filas y se recalcula el número de pacientes
   distintos, que ya no será 71.518.
3. Que los seis códigos han desaparecido de `discharge_disposition_id`.

Sobre el índice: al eliminar filas, el índice de `df` queda con huecos. Se reinicia con
`reset_index(drop=True)` porque a partir de aquí ya no hace falta mantener la
correspondencia posicional con el fichero original (`patient_nbr` ya está pegado) y un
índice continuo evita sorpresas en operaciones posteriores.

In [6]:
codigos_excluir = [11,13,14,19,20,21]

mascara_excluir = df['discharge_disposition_id'].isin(codigos_excluir)
int(mascara_excluir.sum())

2423

In [7]:
filas_marcadas = df[mascara_excluir]

pd.crosstab(filas_marcadas['discharge_disposition_id'], filas_marcadas['readmitted'])

readmitted,<30,>30,NO
discharge_disposition_id,,,
11,0,0,1642
13,19,36,344
14,24,7,341
19,0,0,8
20,0,0,2


In [8]:
# Eliminamos los ingresos de fallecidos y hospice y reiniciamos el índice
df = df[~mascara_excluir].reset_index(drop=True)
df.shape

(99343, 36)

In [9]:
bool(df['discharge_disposition_id'].isin(codigos_excluir).any())

False

In [10]:
df['patient_nbr'].nunique()

69990

### Interpretación del apartado 2

Las tres comprobaciones coinciden con lo previsto:

| Comprobación | Esperado | Obtenido |
|---|---|---|
| Filas que marca la máscara | 2.423 | 2.423 |
| Forma de `df` tras el filtrado | (99.343, 36) | (99.343, 36) |
| Algún código de exclusión sigue presente | `False` | `False` |

La tabla cruzada confirma la evidencia con recuentos, no con porcentajes redondeados: los
tres códigos de fallecimiento presentes (11, 19, 20) son `NO` en **1.652 de 1.652** ingresos,
sin una sola excepción. Un `1,00` redondeado habría sido compatible con unos pocos
readmitidos ocultos; el `0 · 0 · 1642` no deja margen.

### Pacientes distintos: de 71.518 a 69.990

El número de pacientes baja en **1.528**. Como se excluyeron 2.423 ingresos, el reparto es:

- **1.538 ingresos** pertenecían a pacientes cuyo *único* registro en el dataset era el
  ingreso en que fallecieron o pasaron a cuidados paliativos (10 de esos pacientes tenían
  dos ingresos con código de exclusión, por ejemplo un alta a *hospice* seguida del
  fallecimiento). Esos pacientes desaparecen del dataset.
- **885 ingresos** pertenecían a pacientes que conservan ingresos anteriores, dados de alta
  vivos. Esos pacientes siguen en el dataset con sus registros válidos, que es exactamente
  lo que pretendía la decisión de excluir *ingresos* y no *pacientes*.

La cifra de referencia para el resto del notebook, y en particular para la división por
paciente del apartado 7, pasa a ser **69.990 pacientes en 99.343 ingresos**.

### Un residuo técnico: las categorías vacías

Eliminar filas no elimina categorías. `discharge_disposition_id` es de tipo `category`, y
su lista de categorías declaradas sigue teniendo las **26** originales aunque solo **21**
tienen filas: los códigos 11, 13, 14, 19 y 20 siguen existiendo como categorías con cero
ingresos. `pd.crosstab` las ha ignorado, pero otras operaciones no lo hacen (`groupby`
con `observed=False`, `value_counts`, o cualquier codificación *one-hot* generaría columnas
de ceros). Se resuelve al principio del apartado 3, que es donde se revisa la lista de
categorías de cada variable.


## 3 · Agrupación de categorías poco frecuentes y de alta cardinalidad

### Por qué agrupar

El EDA dejó dos problemas distintos con las variables categóricas, y ambos tienen la misma
consecuencia para un modelo: categorías con tan pocos ingresos que **no es posible aprender
nada fiable de ellas**, y que además multiplican el número de columnas en la codificación.

1. **Categorías raras en variables de cardinalidad baja o media.** `admission_type_id`
   tenía en el EDA códigos con 10 y 21 ingresos; `discharge_disposition_id`, `admission_source_id` y
   `payer_code` tienen varias categorías por debajo de 100. Un porcentaje de readmisión
   calculado sobre 10 ingresos es ruido (el EDA lo vio con `acarbose`: 13 ingresos
   ajustados daban un 23 % sin significado). La decisión del apartado 5.6 del notebook 03
   fue agruparlas en una categoría **`Otros`** con un umbral de **100 ingresos**.

2. **Alta cardinalidad.** `diag_1`, `diag_2` y `diag_3` tienen 716, 748 y 789 códigos
   CIE-9 distintos, y `medical_specialty` 73 especialidades. Codificarlas tal cual con
   *one-hot* añadiría más de 2.300 columnas, casi todas con un puñado de unos. Para los
   diagnósticos la alternativa estándar en la literatura clínica (y la que usaron Strack
   et al., 2014, con este mismo dataset) es agrupar los códigos en **capítulos CIE-9**
   (circulatorio, respiratorio, digestivo, diabetes, lesiones, ...), que reducen ~750
   valores a menos de 20 conservando el significado clínico.

El apartado sigue este orden, de lo sencillo a lo complejo:

- **3.1** Eliminar las categorías vacías que dejó el apartado 2.
- **3.2** `Otros` en `admission_type_id`, `discharge_disposition_id`, `admission_source_id`
  y `payer_code`.
- **3.3** Capítulos CIE-9 para `diag_1`, `diag_2` y `diag_3`.
- **3.4** `medical_specialty`.

### 3.1 · Categorías vacías

Antes de agrupar nada hay que dejar las listas de categorías en un estado coherente: si
`discharge_disposition_id` sigue declarando los códigos 11, 13, 14, 19 y 20 con cero
ingresos, cualquier recuento por categoría los mostrará como filas a 0 y la regla de
`Otros` (menos de 100 ingresos) los atraparía como si fueran categorías reales.

pandas ofrece el método `.cat.remove_unused_categories()`, que devuelve la misma Serie con
la lista de categorías reducida a las que tienen al menos un valor. No cambia ningún dato:
solo la lista declarada.

**Qué se comprueba:** que el número de categorías declaradas pasa de **26 a 21** y que
coincide con el número de valores distintos presentes (`nunique()`). Es la única variable
afectada, porque es la única en la que se han eliminado filas por su valor; en el resto,
las categorías se construyeron a partir de los datos al cargar y todas tienen ingresos.


In [11]:
df['discharge_disposition_id'] = df['discharge_disposition_id'].cat.remove_unused_categories()

len(df['discharge_disposition_id'].cat.categories)

21

**Resultado 3.1:** las categorías declaradas pasan de 26 a **21**, que coincide con los 21
valores distintos presentes. La lista vuelve a ser coherente con los datos.

### 3.2 · Categorías poco frecuentes → `Otros`

#### Qué variables y qué umbral

El apartado 5.6 del notebook 03 fijó la regla: en las cuatro variables de cardinalidad baja o
media con códigos raros, toda categoría con **menos de 100 ingresos** se agrupa en una
etiqueta común, `Otros`. El umbral no es arbitrario: por debajo de 100 casos, un porcentaje
de readmisión oscila varios puntos con que cambien dos o tres pacientes, y el EDA ya
comprobó que con esos tamaños no se puede concluir nada (los 13 ingresos de `acarbose`
ajustado daban un 23 % sin significado).

Recuentos sobre los 99.343 ingresos actuales:

| Variable | Categorías | Con < 100 ingresos | Ingresos que van a `Otros` | Categorías después |
|---|---|---|---|---|
| `admission_type_id` | 8 | 2 (códigos 7 y 4: 18 y 10) | 28 | 7 |
| `discharge_disposition_id` | 21 | 8 (15, 24, 9, 17, 16, 10, 27, 12: de 63 a 3) | 171 | 14 |
| `admission_source_id` | 17 | 7 (8, 22, 10, 14, 11, 25, 13: de 15 a 1) | 42 | 11 |
| `payer_code` | 18 | 4 (`OT` 92, `MP` 79, `SI` 55, `FR` 1) | 227 | 15 |

`Otros` significa **"valor conocido pero infrecuente"**. Es distinto de `Unknown` (en
`payer_code`) y de los códigos `NULL` / `Not Available` / `Not Mapped` de los `*_id`, que
significan **"valor desconocido"**. Son dos ausencias de información diferentes y se mantienen
separadas: fundir un pagador raro con un pagador desconocido mezclaría cosas que no tienen
nada que ver.

#### Una consecuencia que conviene ver antes de hacerlo

En dos de las cuatro variables, el propio `Otros` queda por debajo del umbral: 28 ingresos en
`admission_type_id` y 42 en `admission_source_id`. Agrupar categorías raras no garantiza que
el grupo resultante deje de ser raro. Se mantiene igualmente, por dos razones:

- El objetivo de la regla no es que toda categoría sea "aprendible", sino **no tener varias
  columnas con un puñado de unos** tras la codificación. Un `Otros` de 28 es una columna
  casi vacía; los dos códigos originales eran dos.
- La alternativa, fundir esos ingresos con la categoría mayoritaria o con los códigos de
  valor desconocido, inventaría un dato: un ingreso por trauma no es un ingreso de urgencias
  ni un ingreso de origen desconocido.

Queda como limitación documentada: esos 70 ingresos no aportarán señal a un modelo.

#### Por qué una función

La regla es idéntica para las cuatro variables. Escribirla cuatro veces multiplica por cuatro
la probabilidad de un error de copia y esconde la regla en cuatro sitios. Una función
`agrupar_raras(serie, umbral, etiqueta)` la deja escrita una vez, con el umbral como
parámetro explícito, y se aplica en un bucle.

#### La trampa del tipo `category`

Una Serie de tipo `category` **rechaza cualquier valor que no esté en su lista de
categorías**: asignar `'Otros'` directamente da `TypeError`. La secuencia correcta es la
inversa del apartado 3.1: primero `cat.add_categories('Otros')`, después sustituir los
valores raros, y por último `cat.remove_unused_categories()` para retirar de la lista los
códigos que ya no tienen filas. pandas admite mezclar el texto `'Otros'` con las categorías
numéricas de los `*_id`.

Hay un segundo detalle de tipos: la celda 3 solo declaró como `category` los `*_id` y los
`diag_*`; `payer_code` sigue siendo `object`, y `object` no tiene `.cat`. La función
convierte a `category` en su primera línea para no depender de cómo venga la columna
(para una Serie que ya es `category` la conversión no cambia nada). Salió a la luz al
ejecutarla: falló en la cuarta variable tras funcionar en las tres primeras.

#### Qué se comprueba

Para cada variable, tras aplicar la función: el número de categorías declaradas coincide con
la columna "Categorías después" de la tabla, y el recuento de `Otros` coincide con "Ingresos
que van a `Otros`". Y, al final, que en ninguna de las cuatro variables queda una categoría
con menos de 100 ingresos **aparte de `Otros`**.


In [12]:
def agrupar_raras(serie, umbral=100, etiqueta='Otros'):
    """Sustituye por `etiqueta` las categorías de `serie` con menos de `umbral` ingresos."""
    serie = serie.astype('category')   # acepta tanto object como category
    recuento = serie.value_counts()
    raras = recuento[recuento < umbral].index

    serie = serie.cat.add_categories(etiqueta)
    serie = serie.mask(serie.isin(raras), etiqueta)
    serie = serie.cat.remove_unused_categories()
    return serie

In [13]:
columnas_otros = ['admission_type_id', 'discharge_disposition_id',
                  'admission_source_id', 'payer_code']

for col in columnas_otros:
    df[col] = agrupar_raras(df[col])
    n_categorias = len(df[col].cat.categories)
    n_otros = (df[col] == 'Otros').sum()
    print(f'{col:26} categorías: {n_categorias:3}   Otros: {n_otros}')             

admission_type_id          categorías:   7   Otros: 28
discharge_disposition_id   categorías:  14   Otros: 171
admission_source_id        categorías:  11   Otros: 42
payer_code                 categorías:  15   Otros: 227


In [14]:
# Comprobación final: ¿queda alguna categoría con menos de 100 ingresos aparte de 'Otros'?
for col in columnas_otros:
    recuento = df[col].value_counts()
    print(f'{col:26}', recuento[recuento < 100].to_dict())                            

admission_type_id          {'Otros': 28}
discharge_disposition_id   {}
admission_source_id        {'Otros': 42}
payer_code                 {}


### Interpretación del apartado 3.2

Las cuatro variables quedan como preveía la tabla: 7, 14, 11 y 15 categorías, con `Otros`
de 28, 171, 42 y 227 ingresos. La comprobación final confirma que **la única categoría por
debajo de 100 ingresos es `Otros`** en `admission_type_id` (28) y `admission_source_id` (42),
las dos que ya se habían anticipado; en `discharge_disposition_id` y `payer_code` no queda
ninguna.

Dos notas para el resto del notebook:

- La función `agrupar_raras` se reutilizará en el 3.4 con `medical_specialty`.
- `Otros` es ahora una categoría más en las cuatro variables; la codificación del apartado 6
  la tratará como cualquier otra.

### 3.3 · Códigos de diagnóstico → capítulos CIE-9

#### El problema

`diag_1`, `diag_2` y `diag_3` contienen el diagnóstico principal y dos secundarios de cada
ingreso, como códigos **CIE-9** (*International Classification of Diseases, 9ª revisión*):
`250.83` es diabetes con complicación, `428` insuficiencia cardiaca, `V58` un contacto con
servicios sanitarios (por ejemplo, quimioterapia), `E909` una causa externa. Tras el
apartado 2 quedan **715, 747 y 786 códigos distintos** en las tres columnas.

Codificarlos tal cual es inviable: *one-hot* añadiría unas 2.250 columnas, casi todas con
decenas de unos, y la regla de `Otros` del 3.2 no sirve porque la inmensa mayoría de los
códigos son raros: se llevaría casi todo a `Otros` y destruiría la información clínica.

#### La solución: agrupar por capítulo

La CIE-9 está organizada por **rangos numéricos que corresponden a sistemas del organismo**:
390-459 es el aparato circulatorio, 460-519 el respiratorio, 520-579 el digestivo, etc.
Agrupar cada código en su capítulo reduce ~750 valores a un puñado **sin perder el
significado clínico**: "insuficiencia cardiaca" y "fibrilación auricular" pasan a ser
"circulatorio", que es la información que un modelo puede aprovechar con 99.000 ingresos.

Se sigue el esquema de **Strack et al. (2014)**, el artículo que introdujo este dataset, que
agrupó los diagnósticos en nueve grupos. Usar el mismo esquema tiene una ventaja concreta:
sus proporciones publicadas sirven para comprobar que estamos leyendo los códigos igual
que ellos.

| Capítulo | Códigos CIE-9 |
|---|---|
| `Circulatorio` | 390-459 y 785 (síntomas cardiovasculares) |
| `Respiratorio` | 460-519 y 786 (síntomas respiratorios) |
| `Digestivo` | 520-579 y 787 (síntomas digestivos) |
| `Diabetes` | 250.xx |
| `Lesiones` | 800-999 (lesiones y envenenamientos) |
| `Musculoesquelético` | 710-739 |
| `Genitourinario` | 580-629 y 788 (síntomas genitourinarios) |
| `Neoplasias` | 140-239 |
| `Otro` | el resto de códigos numéricos, y los códigos `V` y `E` |
| `Unknown` | sin diagnóstico registrado (los NaN conservados en la limpieza: 20, 356 y 1.419) |

Tres decisiones dentro del esquema:

- Los códigos **`V` y `E`** (factores que influyen en el estado de salud y causas externas)
  no son enfermedades y van a `Otro`, como en el artículo. Son 1.633 en `diag_1` pero
  5.019 en `diag_3`.
- Los **NaN** pasan a `Unknown`, la misma etiqueta que usan `race`, `payer_code` y
  `medical_specialty` para "valor no registrado". Es coherente con la decisión de la
  limpieza de representar la ausencia en vez de imputarla.
- `Otro` (singular) es un capítulo clínico residual y **no es lo mismo** que el `Otros`
  del 3.2 (categorías conocidas pero infrecuentes). Se mantienen nombres distintos para
  que no se confundan.

#### Qué se comprueba

1. Que cada columna queda con **10 categorías** y que su suma es 99.343 (ningún código se
   ha quedado sin capítulo).
2. Que la distribución de `diag_1` se parece a la publicada por Strack et al.:
   circulatorio ~30 %, diabetes ~9 %, respiratorio ~14 %.
3. Que no queda ningún capítulo por debajo de 100 ingresos aparte de `Unknown` en `diag_1`
   (20 ingresos), que se acepta porque representa una ausencia real, no una categoría rara.


In [15]:
def capitulo_cie9(codigo):
    """Traduce un código CIE-9 a su capítulo clínico (esquema de Strack et al., 2014)."""
    if pd.isna(codigo):
        return 'Unknown'

    codigo = str(codigo)

    # Diabetes: 250.xx. Se resuelve aquí porque es donde están casi todos los decimales
    if codigo.startswith('250'):
        return 'Diabetes'

    # Códigos V y E: no son enfermedades. Antes de float(), que fallaría con ellos
    if codigo[0] in ('V', 'E'):
        return 'Otro'

    n = float(codigo)

    # Códigos de síntomas, que la CIE-9 numera lejos del aparato al que corresponden
    sintomas = {785: 'Circulatorio', 786: 'Respiratorio',
                787: 'Digestivo', 788: 'Genitourinario'}
    if n in sintomas:
        return sintomas[n]

    if 140 <= n <= 239:
        return 'Neoplasias'
    if 390 <= n <= 459:
        return 'Circulatorio'
    if 460 <= n <= 519:
        return 'Respiratorio'
    if 520 <= n <= 579:
        return 'Digestivo'
    if 580 <= n <= 629:
        return 'Genitourinario'
    if 710 <= n <= 739:
        return 'Musculoesquelético'
    if 800 <= n <= 999:
        return 'Lesiones'

    return 'Otro'


# Un caso de cada rama antes de soltarla sobre 99.343 filas
for c in ['428', '250.83', 'V58', 'E909', '786', '365.44', np.nan]:
    print(f'{str(c):>8} → {capitulo_cie9(c)}')

     428 → Circulatorio
  250.83 → Diabetes
     V58 → Otro
    E909 → Otro
     786 → Respiratorio
  365.44 → Otro
     nan → Unknown


In [16]:
columnas_diag = ['diag_1', 'diag_2', 'diag_3']

for col in columnas_diag:
    # .astype(object) antes de .apply(): sobre una columna 'category' pandas no pasa los NaN
    # por la función, y los 20 + 356 + 1.419 diagnósticos ausentes se quedarían sin capítulo
    df[col] = df[col].astype(object).apply(capitulo_cie9).astype('category')

df['diag_1'].value_counts()


diag_1
Circulatorio          29681
Otro                  17793
Respiratorio          13934
Digestivo              9333
Diabetes               8661
Lesiones               6853
Genitourinario         5002
Musculoesquelético     4935
Neoplasias             3131
Unknown                  20
Name: count, dtype: int64

In [17]:
for col in columnas_diag:
    print(f'{col}   categorías: {list(df[col].cat.categories)}  {len(df[col].cat.categories)}   suma: {df[col].value_counts().sum()}')


diag_1   categorías: ['Circulatorio', 'Diabetes', 'Digestivo', 'Genitourinario', 'Lesiones', 'Musculoesquelético', 'Neoplasias', 'Otro', 'Respiratorio', 'Unknown']  10   suma: 99343
diag_2   categorías: ['Circulatorio', 'Diabetes', 'Digestivo', 'Genitourinario', 'Lesiones', 'Musculoesquelético', 'Neoplasias', 'Otro', 'Respiratorio', 'Unknown']  10   suma: 99343
diag_3   categorías: ['Circulatorio', 'Diabetes', 'Digestivo', 'Genitourinario', 'Lesiones', 'Musculoesquelético', 'Neoplasias', 'Otro', 'Respiratorio', 'Unknown']  10   suma: 99343


In [18]:
(df['diag_1'].value_counts(normalize=True) * 100).round(2)


diag_1
Circulatorio          29.88
Otro                  17.91
Respiratorio          14.03
Digestivo              9.39
Diabetes               8.72
Lesiones               6.90
Genitourinario         5.04
Musculoesquelético     4.97
Neoplasias             3.15
Unknown                0.02
Name: proportion, dtype: float64

### Interpretación del apartado 3.3

Las tres columnas quedan con **10 capítulos y 99.343 ingresos cada una**: ningún código se ha
quedado sin clasificar. La reducción es la buscada: de 715, 747 y 786 valores distintos a 10.
En la codificación del apartado 6 eso son 30 columnas en total en vez de las ~2.250 que habría
producido un *one-hot* sobre los códigos originales.

#### La distribución coincide con la publicada

| Capítulo | `diag_1` | Strack et al. (2014) |
|---|---|---|
| Circulatorio | 29,88 % | ~30 % |
| Respiratorio | 14,03 % | ~14 % |
| Diabetes | 8,72 % | ~9 % |

Coincidir con el artículo que introdujo el dataset confirma que los rangos y las excepciones
(785-788, códigos `V` y `E`) están bien aplicados. No es una comprobación estética: un error en
un rango habría desplazado miles de ingresos a otro capítulo sin producir ningún aviso.

#### Qué dicen las tres columnas juntas

| Capítulo | `diag_1` | `diag_2` | `diag_3` |
|---|---|---|---|
| Circulatorio | 29,88 | 31,36 | 29,80 |
| **Diabetes** | **8,72** | **12,79** | **17,09** |
| Respiratorio | 14,03 | 10,46 | 7,05 |
| Digestivo | 9,39 | 4,12 | 3,88 |
| Lesiones | 6,90 | 2,40 | 1,91 |
| Otro | 17,91 | 26,20 | 28,78 |
| `Unknown` | 0,02 | 0,36 | 1,43 |

El gradiente de `Diabetes` es el hallazgo más claro: **8,72 % como diagnóstico principal pero
17,09 % como tercero**. En una cohorte en la que todos los pacientes son diabéticos, la diabetes
rara vez es el motivo del ingreso; casi siempre es la comorbilidad que acompaña a otra cosa.
Lo simétrico ocurre con los capítulos agudos (`Respiratorio` 14,03 → 7,05; `Lesiones` 6,90 → 1,91):
el diagnóstico principal recoge **el episodio que trae al paciente**, y los secundarios **el terreno
crónico sobre el que ocurre**. `Circulatorio` es la excepción: ronda el 30 % en las tres posiciones,
porque es a la vez causa frecuente de ingreso y comorbilidad casi universal en esta población.

Esto tiene una consecuencia para el modelo: las tres columnas **no son intercambiables ni
redundantes**. Cada una aporta información de distinta naturaleza y las tres se conservan.

#### Hallazgo metodológico: `.apply()` no pasa los NaN en una columna `category`

El primer intento devolvió **9 capítulos en vez de 10**, sin `Unknown`, y sumas de 99.323, 98.987
y 97.924 en vez de 99.343. Las diferencias eran exactamente 20, 356 y 1.419: los NaN de cada
columna. La función tenía su rama `if pd.isna(codigo): return 'Unknown'` correctamente escrita,
pero **nunca se ejecutaba**: sobre una columna de tipo `category`, pandas (2.3.3) no pasa los
valores ausentes por la función y los devuelve intactos. Y `diag_1/2/3` se habían convertido a
`category` en el apartado 1.

La solución es `.astype(object)` antes de `.apply()`, que devuelve la columna a texto plano el
tiempo justo para que la función recorra todos los valores.

Conviene anotar la alternativa que **no** hay que usar: `.astype(str)` también recorre los NaN,
pero los convierte en la cadena `'nan'`, que la función clasifica como `Otro` sin producir ningún
error. Los 1.419 diagnósticos ausentes de `diag_3` habrían quedado disfrazados de capítulo
clínico real. Es un fallo peor que el original, porque ninguna comprobación lo habría delatado.

El episodio confirma el valor de las comprobaciones: el error no lo detectó nadie leyendo el
código, sino la cuenta de categorías y la suma de filas.

#### Una categoría por debajo del umbral

`Unknown` tiene **20 ingresos en `diag_1`**, por debajo del umbral de 100 del apartado 3.2. Se
conserva igualmente, y por la misma razón que en el resto del proyecto: no es una categoría rara
entre otras posibles, sino la representación de una **ausencia real**. Agruparla con `Otro`
mezclaría "no se registró el diagnóstico" con "el diagnóstico no pertenece a ningún capítulo
con nombre propio", que son cosas distintas.

### 3.4 · La especialidad del servicio → `medical_specialty`

#### La situación

`medical_specialty` indica el servicio del médico que ingresó al paciente. Es la última variable
de alta cardinalidad que queda: **73 valores distintos** tras las exclusiones del apartado 2.

No es el mismo problema que el de los diagnósticos, y por eso no lleva el mismo tratamiento:

| | `diag_1/2/3` | `medical_specialty` |
|---|---|---|
| Valores distintos | 715-786 | 73 |
| Categorías con <100 ingresos | 581 de 715 (11 % de los pacientes) | **44 de 73 (0,92 %)** |
| ¿Existe una taxonomía oficial? | Sí, los capítulos CIE-9 | No |

Aquí las categorías raras son **muchas en número pero insignificantes en volumen**: las 44
especialidades con menos de 100 ingresos suman 911 pacientes, menos del 1 % del dataset. Es
exactamente el caso para el que se escribió `agrupar_raras` en el 3.2, así que se reutiliza esa
función con el mismo umbral de 100, sin inventar nada nuevo.

#### La distribución real

| Especialidad | Ingresos | % |
|---|---|---|
| `Unknown` | 48.616 | 48,94 |
| `InternalMedicine` | 14.237 | 14,33 |
| `Emergency/Trauma` | 7.419 | 7,47 |
| `Family/GeneralPractice` | 7.252 | 7,30 |
| `Cardiology` | 5.279 | 5,31 |
| `Surgery-General` | 3.059 | 3,08 |
| … 67 más | | |

Dos cosas que conviene mirar de frente antes de ejecutar nada:

- **Casi la mitad del dataset es `Unknown`.** Se conserva como categoría propia, según la decisión
  ya cerrada del notebook 02. Pero el EDA (apartado 5.2) demostró con `admission_type_id` y
  `payer_code` que estos huecos identifican **de qué hospital procede el registro**, no qué le
  pasa al paciente. Con un 49 % de ausencias, `medical_specialty` es la variable donde ese riesgo
  es mayor, y habrá que pesarlo al seleccionar las predictoras del apartado 4.
- **Las cinco primeras categorías con nombre cubren el 88 % de lo registrado.** La cola larga
  aporta poco, que es justo lo que el umbral resuelve.

#### Qué comprobar

1. Que las categorías bajan de 73 a **30**: las 29 con 100 ingresos o más, más `Otros`.
2. Que `Otros` reúne **911 ingresos**, cifra que ya conocemos y que sirve de control.
3. Que la suma sigue siendo 99.343.

In [19]:
# Mismo umbral y misma función que en el 3.2: no hace falta nada nuevo
df['medical_specialty'] = agrupar_raras(df['medical_specialty'])

df['medical_specialty'].value_counts()

medical_specialty
Unknown                              48616
InternalMedicine                     14237
Emergency/Trauma                      7419
Family/GeneralPractice                7252
Cardiology                            5279
Surgery-General                       3059
Nephrology                            1539
Orthopedics                           1392
Orthopedics-Reconstructive            1230
Radiologist                           1121
Otros                                  911
Pulmonology                            854
Psychiatry                             853
Urology                                682
ObstetricsandGynecology                669
Surgery-Cardiovascular/Thoracic        642
Gastroenterology                       538
Surgery-Vascular                       525
Surgery-Neuro                          462
PhysicalMedicineandRehabilitation      391
Oncology                               319
Pediatrics                             253
Neurology                           

In [20]:
n_categorias = len(df['medical_specialty'].cat.categories)
n_otros = (df['medical_specialty'] == 'Otros').sum()
suma = df['medical_specialty'].value_counts().sum()

print(f'categorías: {n_categorias}   Otros: {n_otros}   suma: {suma}')

categorías: 30   Otros: 911   suma: 99343


### Interpretación del apartado 3.4

Las tres comprobaciones salen como se había previsto: **30 categorías** (las 29 con 100 ingresos
o más, más `Otros`), `Otros` reúne **911 ingresos** y la suma sigue siendo 99.343.

El coste de la operación es mínimo y conviene verlo con números: se han fundido **44 categorías
en una sola**, y esas 44 representaban el **0,92 %** de los pacientes. A cambio, la variable pasa
de 73 columnas a 30 en la codificación. Compárese con `diag_1`, donde la misma regla habría
afectado al 11 % de los pacientes: es la diferencia entre una cola larga irrelevante y una
distribución genuinamente dispersa, y es la razón por la que cada una llevó un tratamiento
distinto.

`Otros` (911) queda además por encima del umbral, así que no reproduce el problema que pretendía
resolver, cosa que sí ocurre en `admission_type_id` (28) y `admission_source_id` (42).

#### La reserva sobre esta variable

`Unknown` son **48.616 ingresos, el 48,94 %**: casi la mitad del dataset no tiene registrado el
servicio de ingreso. Se conserva como categoría, en coherencia con el resto del proyecto, pero
con una advertencia que el EDA dejó documentada: en `admission_type_id` y `payer_code` se
comprobó que los huecos **identifican al hospital que no rellenaba ese campo**, no a un tipo de
paciente. Con un 49 % de ausencias, `medical_specialty` es la variable donde ese riesgo es mayor.

No se elimina por eso —el 51 % restante sí informa, y `InternalMedicine`, `Emergency/Trauma` o
`Cardiology` describen contextos clínicos muy distintos—, pero queda anotado: si en el futuro un
modelo concede mucha importancia a esta variable, lo primero que hay que descartar es que esté
aprendiendo el centro de procedencia.

### Conclusiones del apartado 3

Ocho variables categóricas han pasado por este apartado. El resultado conjunto:

| Variable | Antes | Después | Cómo |
|---|---|---|---|
| `admission_type_id` | 8 | 7 | `Otros` (28) |
| `discharge_disposition_id` | 26 declaradas / 21 reales | 14 | categorías vacías + `Otros` (171) |
| `admission_source_id` | 17 | 11 | `Otros` (42) |
| `payer_code` | 18 | 15 | `Otros` (227) |
| `diag_1` | 715 | 10 | capítulos CIE-9 |
| `diag_2` | 747 | 10 | capítulos CIE-9 |
| `diag_3` | 786 | 10 | capítulos CIE-9 |
| `medical_specialty` | 73 | 30 | `Otros` (911) |
| **Total** | **2.390** | **107** | |

En términos de la codificación que viene en el apartado 6, esas ocho variables habrían aportado
unas **2.390 columnas** y ahora aportarán **107**. Ningún ingreso se ha perdido por el camino: las
comprobaciones de cada subapartado confirman que la suma sigue siendo 99.343.

#### Lo que este apartado enseña sobre el método

Los tres problemas parecían el mismo —"hay demasiadas categorías"— y han necesitado tres
soluciones distintas:

- **Categorías vacías** (3.1): un residuo técnico de haber filtrado filas. No es una decisión
  analítica, es limpieza: `remove_unused_categories()`.
- **Cola larga irrelevante** (3.2 y 3.4): muchas categorías que suman muy pocos pacientes. El
  umbral de 100 las agrupa sin perder casi nada (0,92 % en `medical_specialty`).
- **Dispersión genuina** (3.3): 715 códigos en los que *ninguno* es mayoritario y la cola larga
  representa al 11 % de los pacientes. Aquí el umbral habría destruido información, y la solución
  tuvo que venir de fuera de los datos: la taxonomía CIE-9 y el esquema de Strack et al.

Aplicar la regla del 3.2 a los diagnósticos habría sido el error cómodo: una sola función para
todo, ejecutada por rutina. La comprobación de que 581 de 715 códigos quedaban por debajo del
umbral es lo que obligó a buscar otra vía.

#### El apartado deja una cosa anotada

`Otros` y `Otro` conviven ahora en el dataset y **no significan lo mismo**: `Otros` son categorías
conocidas pero infrecuentes; `Otro` es un capítulo clínico residual que incluye códigos muy
frecuentes (celulitis, 2.029 ingresos). Los nombres se mantienen distintos a propósito. En la
codificación del apartado 6 generarán columnas separadas por variable, así que no hay riesgo de
que se mezclen, pero conviene recordarlo al interpretar un futuro modelo.

## 4 · Eliminación de variables sin señal

### Por qué se eliminan variables aquí y no antes

La limpieza (notebook 02) eliminó 15 columnas por razones **estructurales**: identificadores,
columnas casi vacías o con una categoría por encima del 99 %. Eran decisiones que se podían tomar
sin saber nada sobre la readmisión.

Este apartado es distinto: aquí se elimina porque el EDA **demostró que la variable no discrimina**.
Es una decisión analítica y cada eliminación necesita su evidencia. El criterio es el mismo de todo
el notebook 03: comparar la tasa de `<30` de cada categoría con la global —que tras las exclusiones
del apartado 2 es del **11,39 %**— y mirar el tamaño `N` antes que el porcentaje.

### Las cinco candidatas

**`gender`** — 0,16 puntos de diferencia sobre dos grupos enormes:

| | N | `<30` |
|---|---|---|
| `Female` | 53.457 | 11,46 % |
| `Male` | 45.886 | 11,30 % |

Con 99.343 ingresos, una diferencia real se vería. No la hay.

**`race`** — los dos grupos que cubren el 94 % del dataset son indistinguibles:

| | N | `<30` |
|---|---|---|
| `Caucasian` | 74.220 | 11,53 % |
| `AfricanAmerican` | 18.772 | 11,45 % |
| `Unknown` | 2.234 | **8,42 %** |
| `Hispanic` | 2.017 | 10,51 % |
| `Other` | 1.472 | 9,78 % |
| `Asian` | 628 | 10,35 % |

La única categoría que se desvía de verdad es `Unknown`, y el EDA ya demostró (5.2) que **eso no es
una característica del paciente**: los `Unknown` tienen la mitad de `number_inpatient` que el resto,
es decir, son pacientes con poco historial previo *en esta red hospitalaria*. Su `NO` es un
artefacto de que el dataset solo registra reingresos dentro de la misma red.

Dicho de otro modo: la única variación que presenta `race` procede de cómo se recogieron los datos,
no de los pacientes. Es el mejor argumento para eliminarla, y de paso evita que un futuro modelo
use una variable sensible para aprender un sesgo de registro.

**Tres fármacos no evaluables** — `nateglinide`, `acarbose` y `glyburide-metformin`. La limpieza los
conservó porque tenían cientos de casos, pero el EDA (5.4) mostró que esos casos están casi todos en
`Steady`, y que las categorías informativas —las de ajuste de dosis— son diminutas:

| Fármaco | `Up` | `Down` | `Steady` | `<30` en `Steady` |
|---|---|---|---|---|
| `nateglinide` | 24 | 11 | 654 | 11,93 % |
| `acarbose` | 10 | 3 | 292 | 8,56 % |
| `glyburide-metformin` | 8 | 6 | 684 | 11,26 % |

Los porcentajes que salen de esas celdas son ruido: el 33,33 % de `acarbose` en `Down` es **un
paciente de tres**, y el 0 % de `glyburide-metformin` en `Up` sale de ocho. El hallazgo del 5.3 fue
que en `insulin` lo que discrimina es *el ajuste de dosis*; en estos tres fármacos el ajuste
sencillamente no se puede medir.

### Lo que NO se elimina, y por qué

- **`payer_code`** (débil, 40 % `Unknown`) y **`change`** (débil, sospechosa de duplicar a
  `insulin`): el apartado 6.2 del EDA comprobó que **no son redundantes** con `age` ni con `insulin`
  respectivamente. Débil no es lo mismo que nulo, y ninguna duplica a otra.
- **`medical_specialty`**, pese al 49 % de `Unknown`, por lo argumentado en el 3.4.
- **Los ocho fármacos restantes**: aunque solo `metformin` e `insulin` tienen señal propia, el resto
  no se ha demostrado inútil, solo poco informativo. Eliminar por "parece que no aporta" sería justo
  el tipo de decisión por rutina que este proyecto evita.

### Qué código hace falta

Una lista con las cinco columnas y un `df.drop(columns=...)`. Después, comprobar que `df` pasa de
**36 a 31 columnas** y que conserva las 99.343 filas.

In [21]:
# Las cinco variables que el EDA demostró que no discriminan
variables_sin_senal = ['gender', 'race',
                       'nateglinide', 'acarbose', 'glyburide-metformin']

df = df.drop(columns=variables_sin_senal)

df.shape

(99343, 31)

### Interpretación del apartado 4

`df` pasa de **36 a 31 columnas** conservando las **99.343 filas**: solo se han eliminado variables,
ningún paciente.

Las 31 columnas se reparten así:

| | Cuántas | Cuáles |
|---|---|---|
| Identificador | 1 | `patient_nbr` (no es predictora; solo sirve para el split del apartado 7) |
| Numéricas | 8 | las mismas desde el principio: `time_in_hospital`, `num_lab_procedures`, `num_procedures`, `num_medications`, `number_outpatient`, `number_emergency`, `number_inpatient`, `number_diagnoses` |
| Categóricas | 21 | `age`, los tres `*_id`, `payer_code`, `medical_specialty`, `diag_1/2/3`, `max_glu_serum`, `A1Cresult`, los 8 fármacos restantes, `change` y `diabetesMed` |
| Objetivo | 1 | `readmitted` |

Es decir, **29 predictoras**.

#### El efecto real de eliminar cinco variables

En número de columnas del futuro modelo, la poda ahorra unas 20: `gender` (2), `race` (6) y los tres
fármacos (4 cada uno). No es una cifra espectacular al lado de las 2.283 que ahorró el apartado 3, y
merece la pena decirlo claramente: **el objetivo aquí no era reducir tamaño, sino quitar ruido**. Una
variable que no discrimina no solo ocupa espacio; ofrece al modelo un sitio donde encontrar patrones
que no existen, y eso es especialmente probable en las categorías diminutas de los tres fármacos,
donde un 33 % calculado sobre tres pacientes parece una señal enorme.

#### Sobre `race`, un matiz que conviene dejar escrito

Se elimina porque el EDA demostró que no discrimina, no por prudencia. Pero el resultado coincide con
lo que hoy se considera buena práctica en modelos clínicos, y por una razón concreta que este
proyecto puede demostrar con sus propios datos: la única variación apreciable de la variable estaba
en `Unknown` (8,42 % frente al 11,39 % global), y el apartado 5.2 del notebook 03 comprobó que esos
pacientes se distinguen por tener **la mitad de `number_inpatient`**, es decir, por llevar poco
historial en esta red hospitalaria. Un modelo que usara `race` no habría aprendido nada sobre etnias:
habría aprendido a detectar registros incompletos.

#### Lo que queda por delante

De las 29 predictoras, el EDA señaló como más prometedoras `number_inpatient` (la mejor numérica, con
un cociente de 3,2× y cobertura del 50 % de los pacientes), `discharge_disposition_id` (la mejor
categórica) e `insulin` (la mejor de cardinalidad baja). El resto acompaña.

El dataset ya tiene la forma definitiva en cuanto a **filas y columnas**. Lo que falta son
transformaciones sobre su contenido: decidir cómo se formula el objetivo (apartado 5), traducir las
categorías a números (apartado 6) y partir en `train` y `test` sin mezclar pacientes (apartado 7).

## 5 · Formulación de la variable objetivo

### El problema

`readmitted` tiene tres valores —`NO`, `>30` y `<30`— pero el proyecto plantea desde el notebook 01
una pregunta concreta: **¿qué pacientes volverán a ingresar antes de 30 días?** Esa ventana no es
arbitraria: la readmisión a 30 días es el indicador que los sistemas de salud usan como medida de
calidad asistencial, porque suele reflejar un alta prematura o un seguimiento insuficiente.

Hay que elegir entre mantener las tres clases o reducir el problema a una pregunta binaria (`<30`
frente a todo lo demás), y la elección debe ser explícita: condiciona las métricas, la
interpretación y la utilidad clínica del modelo.

### Qué es realmente `>30`

Antes de decidir conviene mirar cómo se comporta la clase intermedia frente a los dos mejores
predictores que identificó el EDA:

| | `NO` | `>30` | `<30` |
|---|---|---|---|
| % con ≥1 ingreso previo | 23,38 | **42,80** | 49,71 |
| % dado de alta a casa | 63,04 | **60,61** | 49,51 |

`>30` **cambia de bando según la variable**. En historial previo se parece a `<30` (42,80 frente a
49,71) y no a `NO` (23,38). En destino al alta se parece a `NO` (60,61 frente a 63,04) y no a `<30`
(49,51).

La lectura clínica es coherente: `number_inpatient` mide **fragilidad crónica**, y tanto los `>30`
como los `<30` son pacientes frágiles; el destino al alta mide **cómo ha ido este ingreso concreto**,
y ahí solo destacan los `<30`. Dicho en una frase: `>30` es el paciente crónico frecuentador **cuyo
alta no fue problemática**.

Este hallazgo no estaba en el notebook 03 y sale de plantearse esta pregunta. Es el argumento de
fondo de la decisión que sigue.

### La decisión: objetivo binario

Se define **`readmitted_30d`**: vale 1 si el paciente reingresó antes de 30 días y 0 en cualquier
otro caso.

**Por qué**

1. **Es la pregunta que el proyecto dice responder.** Está escrita así desde el notebook 01.
   Cambiarla ahora por comodidad estadística sería adaptar la pregunta a la herramienta.
2. **La frontera `NO`/`>30` es casi ciega donde importa.** En destino al alta, 63,04 frente a 60,61:
   un modelo de tres clases gastaría capacidad en separar dos grupos casi idénticos, y se equivocaría
   sin parar en una distinción que no cambia ninguna decisión clínica.
3. **Las métricas son interpretables.** Precision, recall y F1 de una sola clase, en vez de una matriz
   3×3 con promedios macro y micro que hay que explicar antes de poder discutir nada.

**El coste, dicho sin suavizar**

La clase negativa queda **heterogénea**: mete en el mismo saco a 52.527 pacientes con poco historial
y a 35.502 frecuentadores. En `number_inpatient` —la mejor variable numérica del EDA— el 42,80 % de
esa clase negativa se comporta como los positivos, lo que dilutirá parte de su poder discriminante.

Se asume ese coste porque **la información no se pierde**: la fragilidad de los `>30` sigue en el
dataset, en `number_inpatient`, `number_emergency` y `number_diagnoses`. El modelo seguirá viendo a
esos pacientes; simplemente no se le pide que los etiquete aparte.

**Lo que se descartó**

Modelar solo `NO` frente a `<30`, eliminando los `>30`, daría una clase negativa limpia. Pero supone
tirar 35.502 ingresos (el 36 %) y, sobre todo, produce un modelo inútil en la práctica: en el momento
del alta nadie sabe si ese paciente será un `>30`, así que no habría forma de decidir a quién
aplicárselo.

### Consecuencia inmediata para el futuro modelo

El reparto queda en **11.314 positivos (11,39 %) frente a 88.029 negativos (88,61 %)**. El
desbalance se acentúa respecto a las tres clases, y con ello la advertencia que el notebook 01 ya
dejó escrita: **el *accuracy* no sirve como métrica**. Un modelo que prediga siempre 0 acierta el
88,61 % sin haber aprendido nada. La métrica principal será el **recall de la clase positiva** —qué
proporción de los reingresos tempranos se detecta—, acompañado de precision, F1 y la matriz de
confusión.

### Qué código hace falta

Tres pasos, con una comprobación entre medias:

1. Crear `readmitted_30d` a partir de `readmitted`, con 1 para `<30` y 0 para el resto, en formato
   entero (no booleano: los modelos esperan 0/1).
2. **Comprobar la traducción antes de borrar nada**: una tabla cruzada de `readmitted` contra
   `readmitted_30d` debe enseñar que el 1 se corresponde exactamente con los 11.314 `<30`, y que los
   `NO` y `>30` han ido todos al 0.
3. Solo entonces, eliminar `readmitted`. Esto **no es opcional**: si la columna original sobrevive,
   la codificación del apartado 6 la convertiría en columnas predictoras y el modelo tendría la
   respuesta entre los datos de entrada. Es la fuga de datos más elemental que existe.

Al terminar, `df` debe seguir teniendo **99.343 × 31**: se elimina una columna y se añade otra.

In [22]:
df['readmitted_30d'] = (df['readmitted'] == '<30').astype('int')

df['readmitted_30d'].value_counts()

readmitted_30d
0    88029
1    11314
Name: count, dtype: int64

In [23]:

pd.crosstab(df['readmitted'], df['readmitted_30d'], margins=True)

readmitted_30d,0,1,All
readmitted,,,
<30,0,11314,11314
>30,35502,0,35502
NO,52527,0,52527
All,88029,11314,99343


In [24]:
df = df.drop(columns='readmitted')

df.shape

(99343, 31)

### Interpretación del apartado 5

Las tres comprobaciones salen exactas:

| Comprobación | Esperado | Obtenido |
|---|---|---|
| Positivos / negativos | 11.314 / 88.029 | 11.314 / 88.029 |
| Cada valor original en una sola columna | ceros fuera de la diagonal | `<30` → 11.314 en el 1; `>30` y `NO` → 35.502 y 52.527 en el 0; **cero** en el resto |
| Forma final | 99.343 × 31 | 99.343 × 31 |

La variable objetivo es ahora `readmitted_30d`, con un **11,39 % de positivos**. `readmitted` ha
desaparecido del dataset, y con ella la posibilidad de que la respuesta se cuele entre las
variables de entrada.

#### Por qué recuentos y no proporciones

La tabla cruzada se construye con **recuentos**. Con proporciones redondeadas a dos decimales, la
misma comprobación sería incapaz de demostrar que no hay errores: en la fila del 0 podrían esconderse
**hasta 442 pacientes `<30` mal asignados** (casi el 4 % de los positivos) y la casilla seguiría
marcando `0.00`; en la fila del 1, hasta 56 pacientes `>30`. Es la misma lección del apartado 2: un
`0` exacto es una prueba; un `0.00` redondeado, no.

#### La composición de la clase negativa

Los recuentos dan además la cifra que faltaba para medir el coste de la decisión binaria:

| Clase 0 | Ingresos | % de la clase |
|---|---|---|
| `NO` | 52.527 | 59,67 % |
| `>30` | 35.502 | **40,33 %** |

**Dos de cada cinco "negativos" son pacientes que sí volvieron al hospital**, solo que después de
los 30 días. Es la heterogeneidad que el apartado 5 admitió como coste, ahora cuantificada: cualquier
modelo futuro tendrá que separar los `<30` de un grupo que contiene a muchos pacientes parecidos a
ellos. Conviene recordarlo al valorar su rendimiento: una precisión modesta no será necesariamente
un fallo del modelo, sino en parte el reflejo de esta decisión.

## 6 · Codificación de las variables categóricas

### Por qué hay que codificar

Un modelo trabaja con números. Quedan **21 columnas** cuyo contenido es texto —`'Circulatorio'`,
`'Steady'`, `'[70-80)'`— y hay que traducirlas. La traducción no es neutra: **cómo se codifica una
variable afirma algo sobre ella**. Codificar `A1Cresult` como 0, 1, 2, 3 afirma que existe un
gradiente; codificarla en columnas separadas afirma que cada valor es una categoría distinta. Por eso
no se aplica una regla única, sino la que corresponde a la naturaleza de cada variable.

### La regla: la codificación tiene que decir lo mismo que la variable

| Tipo | Variables | Codificación | Columnas |
|---|---|---|---|
| Binarias | `change`, `diabetesMed` | una columna 0/1 | 2 |
| Ordinal | `age` | punto medio del tramo | 1 |
| Nominales | las 18 restantes | *one-hot* | 147 |

**Binarias.** Con dos valores, una columna 0/1 lo dice todo. Un *one-hot* produciría dos columnas
en las que una es exactamente la contraria de la otra.

**`age`, ordinal.** La edad tiene un orden natural, y codificarla en columnas separadas lo tiraría.
Cada tramo se sustituye por su **punto medio** (5, 15, …, 95): conserva el orden y además la unidad,
los años. Pero hay que distinguir dos cosas que se confunden a menudo: **que una variable sea
ordinal no significa que su efecto sea lineal**. Con el dataset actual:

| Tramo | N | `<30` |
|---|---|---|
| `[20-30)` | 1.649 | **14,31 %** |
| `[50-60)` | 17.060 | 9,77 % |
| `[80-90)` | 16.434 | 12,57 % |

El riesgo baja y vuelve a subir. La codificación ordinal es correcta —describe bien la variable—,
pero un futuro modelo lineal tendrá que tener en cuenta esa forma; los modelos de árboles la capturan
por sí solos. Los dos primeros tramos (1,88 % y 5,80 %) tienen 160 y 690 pacientes y no deben pesar
en esta lectura.

**Nominales, *one-hot*.** Cada categoría pasa a ser una columna con 1 si el ingreso la tiene y 0 si
no. Es la codificación que no impone ningún orden, y conviene justificarla en tres casos donde
podría parecer que sí lo hay:

- **Los códigos administrativos** (`*_id`): son etiquetas. El código 6 no es "más" que el 1.
- **`A1Cresult` y `max_glu_serum`**: sus valores (`Norm`, `>7`, `>8`) parecen una escala, y el
  notebook 01 los ordenó así para las gráficas. Pero la evidencia dice otra cosa. En `A1Cresult`,
  `Norm` 9,77 %, `>7` 10,15 % y `>8` 9,94 % son prácticamente iguales, y la categoría que se desvía es
  `Not_Measured` (11,68 %), que **no tiene sitio en ninguna escala** y además es el 83 % de los casos
  (el 95 % en `max_glu_serum`). Lo que importa es que la prueba se haga, no su resultado: una
  codificación ordinal impondría un gradiente que los datos desmienten.
- **Los ocho fármacos**: `No`, `Steady`, `Up`, `Down` parecen una escala de dosis. Pero el EDA (5.3)
  mostró en `insulin` que `Down` (13,90 %) y `Up` (12,99 %) son las dos categorías de más riesgo:
  lo que discrimina es que **se ajuste** la pauta, no la dirección del ajuste. No existe un orden en
  el que `Up` y `Down` queden en extremos opuestos y los dos arriba.

**Se conservan todas las categorías.** Es habitual eliminar una columna por variable (la
*categoría de referencia*), porque es redundante: si todas las demás son 0, la eliminada es 1. Esa
redundancia es un problema para una regresión logística sin regularizar, pero no para los modelos de
árboles ni para los regularizados, y conservar todas las columnas mantiene cada categoría visible por
su nombre. Como el modelo se elegirá en el proyecto siguiente, se deja la decisión allí: eliminar
una columna es una línea; adivinar cuál se eliminó, no.

**Las numéricas no se tocan.** Estandarizarlas (restar la media y dividir por la desviación típica)
es habitual antes de modelar, pero **esas dos cifras se calculan a partir de los datos**. Hacerlo
aquí, antes de separar `train` y `test`, metería información del conjunto de test en el de
entrenamiento. La estandarización pertenece al proyecto de modelado, ajustada solo sobre `train`.

### ¿Por qué codificar antes de la partición?

La pregunta es legítima, porque el apartado 7 insiste en no mezclar información entre `train` y
`test`. La respuesta es que **estas codificaciones no aprenden nada de los datos**: son reglas fijas.
El punto medio de `[70-80)` es 75 se mire el conjunto que se mire. Y hacerlo antes garantiza que
`train` y `test` tengan **exactamente las mismas columnas**; codificarlos por separado es un error
clásico, porque una categoría ausente en uno de los dos produce tablas con columnas distintas.

Hay una salvedad que conviene dejar escrita. El umbral de 100 ingresos del apartado 3 **sí** se
calculó sobre el dataset completo, antes de partir. Es una simplificación habitual y deliberada: lo
único que cruza de `test` a `train` es si una categoría llegaba a 100 casos en total, lo que no
altera de forma apreciable la evaluación de un modelo. Pero es una simplificación, no un procedimiento
estricto, y se recoge como tal.

### Resultado esperado

**99.343 × 160**: las 10 columnas numéricas (8 predictoras, `patient_nbr` y el objetivo), 2 binarias,
1 ordinal y 147 de *one-hot*. Se hace en tres pasos, cada uno con su comprobación.

### 6.1 · Binarias

El mismo recurso del apartado 5: comparar con el valor que significa "sí" y convertir a entero.

| Variable | "Sí" | Recuento esperado del 1 | del 0 |
|---|---|---|---|
| `change` | `'Ch'` | 46.122 | 53.221 |
| `diabetesMed` | `'Yes'` | 76.719 | 22.624 |

Se sobrescriben las dos columnas con su versión 0/1 y se comprueba que los recuentos coinciden con
los de la tabla. Si un valor estuviera mal escrito (`'Yes '`, `'CH'`), todo quedaría en 0 sin ningún
aviso, y esa comprobación es lo único que lo delataría.

In [25]:
df['change'] = (df['change'] == 'Ch').astype('int')
df['diabetesMed'] = (df['diabetesMed'] == 'Yes').astype('int')

display(df['change'].value_counts())
df['diabetesMed'].value_counts()

change
0    53221
1    46122
Name: count, dtype: int64

diabetesMed
1    76719
0    22624
Name: count, dtype: int64

**Resultado 6.1:** los recuentos coinciden exactamente con los originales: `change` tiene **46.122
unos** (el 46,43 % de los ingresos tuvo algún cambio de medicación) y `diabetesMed` **76.719** (el
77,23 % salió con medicación antidiabética). Las dos columnas son ahora enteros 0/1.

**Nota de método.** El primer intento devolvió **solo ceros** en ambas columnas con un código
correcto. La causa fue ejecutar la celda dos veces sin reiniciar el kernel: tras la primera, `change`
ya contenía 0 y 1, y preguntar si un número es igual a `'Ch'` da `False` en todas las filas. Al revisar
el notebook apareció un segundo problema más grave: las celdas del apartado 5 que eliminaban
`readmitted` no se habían ejecutado en esa sesión, así que **la variable objetivo original seguía en
memoria**. En el 6.3 se habría codificado como una predictora más: la fuga de datos del apartado 5,
entrando por la puerta de atrás en un notebook cuyo código era correcto.

La lección es que **la memoria del kernel no es el archivo**. Las celdas que sobrescriben una columna
no se pueden repetir, y la única garantía de que el estado corresponde al notebook leído de arriba
abajo es *Restart and Run All*.

### 6.2 · `age`, ordinal

Cada tramo se sustituye por su punto medio con un diccionario y `.map()`. Los recuentos por tramo que
tienen que conservarse:

| Tramo | Punto medio | Ingresos |
|---|---|---|
| `[0-10)` | 5 | 160 |
| `[10-20)` | 15 | 690 |
| `[20-30)` | 25 | 1.649 |
| `[30-40)` | 35 | 3.764 |
| `[40-50)` | 45 | 9.607 |
| `[50-60)` | 55 | 17.060 |
| `[60-70)` | 65 | 22.059 |
| `[70-80)` | 75 | 25.331 |
| `[80-90)` | 85 | 16.434 |
| `[90-100)` | 95 | 2.589 |

`.map()` tiene una trampa del mismo tipo que las anteriores: si un valor de la columna **no está** en el
diccionario —basta un corchete donde iba un paréntesis—, lo convierte en vacío (`NaN`) sin ningún aviso.
Por eso se comprueba que no queda **ningún** vacío y que los recuentos por punto medio coinciden con la
tabla. Una pista adicional: si todo va bien, la columna resultante es de tipo entero; si sale decimal
(`float`), es que hay algún `NaN`, porque en pandas un entero no puede estar vacío.

Y la misma advertencia del 6.1: tras la transformación, `age` ya contiene números, y volver a aplicar el
diccionario la dejaría entera en `NaN`.

In [26]:
df['age'] = df['age'].map({ '[0-10)': 5, '[10-20)': 15, '[20-30)': 25, '[30-40)': 35, '[40-50)': 45, '[50-60)': 55, '[60-70)': 65, '[70-80)': 75, '[80-90)': 85, '[90-100)': 95 })
df['age'].value_counts().sort_index()

age
5       160
15      690
25     1649
35     3764
45     9607
55    17060
65    22059
75    25331
85    16434
95     2589
Name: count, dtype: int64

**Resultado 6.2:** los diez recuentos coinciden uno a uno con la tabla. La propia salida confirma las
dos comprobaciones sin necesidad de celdas adicionales: `value_counts()` ignora los vacíos, y aun así
los diez recuentos **suman 99.343**, así que ninguna fila ha quedado sin traducir; y el índice aparece
como `5`, `15`, `25` y no como `5.0`, `15.0`, lo que indica que la columna es de tipo entero, algo
imposible si contuviera un solo `NaN`. `age` es ahora la edad aproximada en años.

### 6.3 · Las 18 nominales, *one-hot*

#### Las columnas

Las 18 nominales se escriben **a mano**, agrupadas por tipo, en lugar de obtenerlas por su tipo de
dato con `select_dtypes`. Así queda escrito qué se codifica, sin tener que deducirlo, y los dos riesgos
de una lista manual están cubiertos: una errata en un nombre hace que `get_dummies` falle con un error
explícito, porque esa columna no existe; y olvidar una columna la dejaría como texto, algo que detecta
la segunda comprobación de abajo. Una lista escrita a mano tiene además una ventaja sobre la obtenida
por tipo: **no depende del estado de la memoria**, así que no puede incluir por accidente una columna
como `readmitted`, el caso que casi ocurre en el 6.1.

| Variables | Categorías | Columnas |
|---|---|---|
| `admission_type_id` | 7 | 7 |
| `discharge_disposition_id` | 14 | 14 |
| `admission_source_id` | 11 | 11 |
| `payer_code` | 15 | 15 |
| `medical_specialty` | 30 | 30 |
| `diag_1`, `diag_2`, `diag_3` | 10 cada una | 30 |
| `A1Cresult`, `max_glu_serum` | 4 cada una | 8 |
| 8 fármacos | 4 cada uno | 32 |
| **Total** | | **147** |

#### Aplicar el *one-hot*

`pd.get_dummies()` sustituye cada una de esas columnas por una columna por categoría, con el nombre
`variable_categoría` (`diag_1_Circulatorio`, `insulin_Down`, `admission_type_id_Otros`…). Tres detalles
de la función que determinan que el resultado sea correcto:

- **`dtype=int`**: sin él, pandas rellena con `True`/`False` en vez de 1/0.
- **Devuelve un DataFrame nuevo**: el resultado hay que asignarlo a `df`. Si no, la celda muestra la
  tabla transformada y `df` sigue intacto.
- **Repetir la celda falla ruidosamente.** Tras la primera ejecución las 18 columnas ya no existen,
  así que ejecutarla otra vez produce un error en lugar de no hacer nada. Después de lo ocurrido en el
  6.1, esta vez el error es preferible al silencio.

#### Qué comprobar

1. **Forma: 99.343 × 160** — 31 columnas, menos las 18 nominales, más las 147 nuevas.
2. **No queda ninguna columna de texto**: `select_dtypes` sobre `object` y `category` tiene que devolver
   cero columnas.
3. **La regla del *one-hot***: para una variable cualquiera, cada ingreso tiene exactamente un 1 entre
   sus columnas. Sumando por filas las columnas de, por ejemplo, `A1Cresult`, el resultado tiene que ser
   `1` en los 99.343 ingresos.

In [27]:
nominales = [
    # administrativas
    'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
    'payer_code', 'medical_specialty',
    # diagnósticos (capítulos CIE-9)
    'diag_1', 'diag_2', 'diag_3',
    # pruebas de laboratorio
    'max_glu_serum', 'A1Cresult',
    # fármacos
    'metformin', 'repaglinide', 'glimepiride', 'glipizide',
    'glyburide', 'pioglitazone', 'rosiglitazone', 'insulin',
]


df = pd.get_dummies(df, columns=nominales, dtype='int')

In [28]:
df.shape

(99343, 160)

In [29]:
# Ninguna columna de texto: las 18 nominales se han codificado todas
df.select_dtypes(['category', 'object']).shape

(99343, 0)

In [30]:
# La regla del one-hot: cada ingreso tiene exactamente un 1 entre las columnas de A1Cresult
df.filter(like='A1Cresult_').sum(axis=1).value_counts()

1    99343
Name: count, dtype: int64

### Interpretación del apartado 6

| Comprobación | Esperado | Obtenido |
|---|---|---|
| Forma | 99.343 × 160 | 99.343 × 160 |
| Columnas de texto restantes | 0 | 0 |
| Unos por fila entre las columnas de `A1Cresult` | 1 en todos los ingresos | 1 en los 99.343 |

El dataset es ahora **íntegramente numérico**: las 160 columnas son enteros. Se reparten así:

| Bloque | Columnas |
|---|---|
| Identificador (`patient_nbr`) | 1 |
| Objetivo (`readmitted_30d`) | 1 |
| Numéricas originales | 8 |
| Binarias (`change`, `diabetesMed`) y ordinal (`age`) | 3 |
| *One-hot* de las 18 nominales | 147 |
| **Total** | **160** |

Es decir, **158 columnas predictoras**.

#### Una conexión con el apartado 3.3

La regla de "exactamente un 1 por fila" tiene una condición que no se ve a simple vista: que la
variable **no tenga vacíos**. `get_dummies` no crea ninguna columna para los `NaN`, así que una fila con
la variable vacía queda con **todo ceros**. Si en el 3.3 no se hubiera corregido el fallo de `.apply()`,
los 20, 356 y 1.419 diagnósticos ausentes habrían llegado hasta aquí como filas sin ningún 1 en sus
columnas de diagnóstico: pacientes que el modelo no podría distinguir de nada. Convertirlos en `Unknown`
no servía solo para que cuadraran las cuentas de aquel apartado; era la condición para que este
funcionara.

#### Lo que se deja, a propósito, para el proyecto de modelado

- **El escalado de las numéricas**, porque se calcula a partir de los datos y debe ajustarse solo sobre
  `train`.
- **La categoría de referencia**: si se elige una regresión sin regularizar, habrá que eliminar una
  columna por variable, eligiéndola con criterio clínico (`No` en los fármacos) y no por orden
  alfabético, que en los fármacos eliminaría justamente `Down`, la categoría de más riesgo.
- **La forma del efecto de `age`**, ordinal pero no monótono.
- **Los nombres de las columnas**, que incluyen `>`, `/` y tildes (`A1Cresult_>7`,
  `medical_specialty_Family/GeneralPractice`, `diag_1_Musculoesquelético`). Algunas librerías de
  modelado restringen ciertos caracteres en los nombres de columna; conviene tenerlo presente al
  cargar los datos.